# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

For each record set, we print the `@id`, name, and available fields (with `@id`).

In [ ]:
# Explore the record sets and fields
record_sets = metadata.recordSet  # List of RecordSet objects

if not record_sets:
    print("No record sets found in metadata! Try reloading or check schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'name' in rs:
            print(f"  Name: {rs['name']}")
        fields = rs.get('field', [])
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    Field @id: {f['@id']} | Name: {f.get('name', '-')}")
        else:
            print("  No fields found in this record set.")
        print("---")
    # Save the record set IDs and field IDs for later
    record_set_ids = [rs['@id'] for rs in record_sets]
    # For demonstration: Pick the first record set
    record_set_id = record_set_ids[0]
    # List all field ids for first record set
    field_ids = [f['@id'] for f in record_sets[0].get('field',[])]
else:
    # Just in case no record sets exist
    record_set_ids = []
    record_set_id = None
    field_ids = []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded data for RecordSet {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for RecordSet {rs_id}")

# For future steps, select the first RecordSet with data
main_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes and not dataframes[rs_id].empty:
        main_record_set_id = rs_id
        break
if main_record_set_id is None:
    raise ValueError("No record set with data found.")
main_df = dataframes[main_record_set_id]
print(f"Selected RecordSet ID for analysis: {main_record_set_id}")
print("Columns: ", main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- For this dataset, let's assume the following fields are present (update based on actual schema):
    - Age: numeric field reference by its `@id` (e.g. `age_field_id`)
    - MSI_status: categorical (e.g. `msi_status_field_id`)
    - Anatomical_location: categorical (e.g. `anatomical_location_field_id`)

Below, we demonstrate filtering and normalization using @ids as keys.

In [ ]:
# Replace these IDs with correct values from the overview above
age_field_id = 'age'  # Example field @id
msi_status_field_id = 'MSI_status'  # Example field @id
anatomical_location_field_id = 'Anatomical_location'  # Example field @id

# Update to actual column names if needed
if age_field_id not in main_df.columns:
    print("age_field_id not found in columns. Available columns:", main_df.columns.tolist())

# Filter records with Age > 50
threshold = 50
if age_field_id in main_df.columns:
    filtered_df = main_df[main_df[age_field_id] > threshold]
    print(f"Filtered records with {age_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize Age
    filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
    print(f"Normalized {age_field_id} for filtered records:")
    print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

    # Group by MSI_status and compute mean Age
    if msi_status_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(msi_status_field_id)[age_field_id].mean().reset_index()
        print(f"Grouped data by {msi_status_field_id} (mean age):")
        print(grouped_df.head())

else:
    print(f"{age_field_id} column not present. Available columns: {main_df.columns.tolist()}" )

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the Age distribution and mean Age per MSI status and Anatomical location.

In [ ]:
if age_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    main_df[age_field_id].hist(bins=20)
    plt.title(f"Distribution of {age_field_id}")
    plt.xlabel(age_field_id)
    plt.ylabel("Count")
    plt.show()

    if msi_status_field_id in main_df.columns:
        msigrp = main_df.groupby(msi_status_field_id)[age_field_id].mean()
        msigrp.plot(kind="bar", title=f"Mean {age_field_id} per {msi_status_field_id}")
        plt.ylabel(f"Mean {age_field_id}")
        plt.show()
    
    if anatomical_location_field_id in main_df.columns:
        locgrp = main_df.groupby(anatomical_location_field_id)[age_field_id].mean()
        locgrp.plot(kind="bar", title=f"Mean {age_field_id} per {anatomical_location_field_id}")
        plt.ylabel(f"Mean {age_field_id}")
        plt.show()
else:
    print("No age field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the FAIR² dataset package using Croissant and `mlcroissant`.
- Key clinicopathological variables such as age, MSI status, and anatomical distribution were examined.
- The dataset enables investigation of predictors and characteristics of second primary colorectal cancer in survivors.
- Further analyses can be performed based on detailed field semantics, clinical phenotypes, and molecular markers thanks to standardized Croissant metadata referencing via `@id`.